# test_graph

Notebook test for TaskRouterGraph flow with a fake LLM.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import os
import sys
import types

PROJECT_ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / 'src' / 'task_router_graph').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError('Cannot locate project root')

SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

# Stub langchain_openai to avoid external dependency in notebook test
stub = types.ModuleType('langchain_openai')
class ChatOpenAI:
    def __init__(self, *args, **kwargs):
        pass
    def invoke(self, messages):
        return SimpleNamespace(content='{}')
stub.ChatOpenAI = ChatOpenAI
sys.modules.setdefault('langchain_openai', stub)

os.environ.setdefault('API_KEY_Qwen', 'dummy')

from task_router_graph.graph import TaskRouterGraph


In [ ]:
class FakeLLM:
    def __init__(self, outputs):
        self.outputs = outputs
        self.i = 0

    def invoke(self, messages):
        if self.i >= len(self.outputs):
            raise RuntimeError('FakeLLM outputs exhausted')
        out = self.outputs[self.i]
        self.i += 1
        return SimpleNamespace(content=out)

graph = TaskRouterGraph(config_path='configs/graph.yaml')
graph._llm = FakeLLM([
    '{"action_kind":"observe","tool":"ls","args":{"path":"data/cases"},"reason":"????"}',
    '{"action_kind":"generate_task","task_type":"functest","task_content":"?? anthropic_ver_1 ??????????? headers?body ? assert","reason":"????"}'
])

result = graph.run(case_id='case_graph_nb', user_input='?????????')
assert result['output']['task_type'] == 'functest'
assert result['output']['task_status'] == 'done'
assert len(result['environment']['rounds']) == 1
assert len(result['environment']['rounds'][0]['controller_trace']) == 2
result['output']
